# 01 - Reading and editing an MCNP input

Solutions of the exercises

In [ ]:
# Run this cell first. It sets up the paths used by the whole notebook.
from pathlib import Path

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

%matplotlib inline
print("data folder :", DATA.resolve())
print("output folder:", OUT.resolve())

---

# Exercises

## Exercise 1 - extract the outboard magnet

Suppose you want to extract the outboard magnet. You know that everything that has material 1 and 13 is part of this magnet.

Produce a standalone input file `outboard_magnet.i` in the `outputs` folder that:

1. contains only those cells (you will have to retrieve them knowing that they  must have either material 1 or 13 assigned),
2. has all its cells and surfaces **shifted by +1000**, so the IDs cannot clash with anything
   else later,
3. carries only the materials it actually needs (`M1` and `M13`),
4. has no tallies and no source left in it.

**Hints**
* `extract_cells` takes a `renumber_offsets` argument: a dictionary passed straight on to
  `renumber`, e.g. `{"cells": 1000, "surfs": 1000}`;
* `remove_tallies()` and `remove_sdef()` take no arguments when you want to remove everything;
* replace `outboard.mat_section` with `inp.get_materials_subset([...])`.

In [ ]:
from f4enix import Input

inp = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")

# 1: the cell IDs of the outboard magnet
materials = [1, 13]
outboard_cells = []
for key, cell in inp.cells.items():
    if cell.material in materials:
        outboard_cells.append(key)

# 2: extract, shifting cells and surfaces by 1000
outboard = inp.extract_cells(
    outboard_cells,
    renumber_offsets={"cells": 1000, "surfs": 1000},
)

# 3: the tallies and the source refer to cells that are no longer here
outboard.remove_tallies()
outboard.remove_sdef()

# 4: only two materials are actually used
outboard.mat_section = inp.get_materials_subset(["M1", "M13"])

# 5: write it out
outboard.write(OUT / "outboard_magnet.i")

print("cells    :", list(outboard.cells.keys()))
print("materials:", [m.name for m in outboard.mat_section])

In [ ]:
# --- self-check, do not modify ---
reloaded = Input.from_input(OUT / "outboard_magnet.i")

expected_ids = {str(i) for i in range(1085, 1106)}
expected_ids.update({str(i) for i in range(1002, 1023)})
assert set(reloaded.cells.keys()) == expected_ids, (
    f"expected cells 1085-1105 and 1002-1021, got {sorted(reloaded.cells.keys())}"
)
assert reloaded.tally_keys == [], f"there are still tallies: {reloaded.tally_keys}"
assert "sdef" not in [k.lower() for k in reloaded.other_data], "the source is still there"
assert sorted(m.name for m in reloaded.mat_section) == ["M1", "M13"], (
    f"materials are {[m.name for m in reloaded.mat_section]}"
)
print("Exercise 1 correct")

## Exercise 2 - Split a cell into two and fill them

Repeat the universe insertion of section 8, but with some twists:

1) define a new surface (999) as a plane passing through the origin and perpendicular to the x axis ('PX 0').
2) create a copy of cell number 2 and assign it to 999. Now split both cells with the plane you just created. Cell 2 will include all X-positive points.
3) Put the filler into **both** cell 1
and cell 2 of the sphere model, and use **universe 7** instead of 10. 

Write the result to `outputs/sphere_double_fill.i`.

**Hints**
* the steps are always the same: renumber the filler, give its cell a `u=`, merge, then set
  `fill=` on the target cells;
* an offset of 100 works again;
* `Input.set_param(cell, "fill", 7)` - one call per cell you want to fill.
* All cards have an attribute ``.text``

In [ ]:
sphere2 = Input.from_input(DATA / "sphere" / "Sphere.i")
filler2 = Input.from_input(DATA / "sphere" / "filler.i")

# 1: Create the new required surfaces and cells
sphere2._model.add_surface('999 PX 0')
newcell = sphere2._model.add_cell(sphere2.cells["2"].text)
newcell.set_id(999)

# 2: split the cells
sphere2.add_surface(sphere2.cells["2"], 999)
sphere2.add_surface(sphere2.cells["999"], -999)

# 3: no ID clash after this
filler2.renumber(cells=100, surfs=100)

# 4: the filler cell becomes universe 7
Input.set_param(filler2.cells["101"], "u", 7)
    
# 5: bring it into the sphere model
sphere2.merge(filler2)

# 6: both cells are filled by that universe
for cell_id in ["2", "999"]:
    Input.set_param(sphere2.cells[cell_id], "fill", 7)

# 7: write it out
sphere2.write(OUT / "sphere_double_fill.i")

print(sphere2._model.to_source())

In [ ]:
# --- self-check, do not modify ---
res = Input.from_input(OUT / "sphere_double_fill.i")

assert "101" in res.cells, f"the filler cell is missing, cells are {list(res.cells.keys())}"
assert "999" in res.cells, f"the new cell is missing, cells are {list(res.cells.keys())}"
assert res.cells["101"].universe == 7, f"filler universe is {res.cells['101'].universe}"
for cid in ("999", "2"):
    assert res.cells[cid].fill is not None, f"cell {cid} has no fill card"
    assert res.cells[cid].fill.universe == 7, f"cell {cid} fill is {res.cells[cid].fill.universe}"
assert res.cells["2"].geometry[-1].text == '999', f"last geometry entry for cell 2 is {res.cells['2'].geometry[-1]}"
assert res.cells["999"].geometry[-1].text == '-999', f"last geometry entry for cell 999 is {res.cells['999'].geometry[-1]}"
for cell in ["2", "999"]:
    assert res.cells[cell].geometry[-2].text == ')', f"cell {cell} was not split it seems"
print("Exercise 2 correct")